In [1]:
# Position class

from __future__ import annotations

from dataclasses import dataclass
from functools import lru_cache
from math import sqrt, isclose, cos, sin, isnan, isinf

# from src.main.world_objects.robot_objects.degrees import Degrees


class InvalidPositionError(Exception):
    """Custom exception for invalid position coordinates."""
    pass


@dataclass(frozen=True)
class Position:
    x: float
    y: float

    def __post_init__(self):
        # Check if coordinates are numeric
        if not isinstance(self.x, (int, float)) or not isinstance(self.y, (int, float)) or isnan(self.x) or isnan(self.y) or isinf(self.x) or isinf(self.y):  #
            raise InvalidPositionError(
                f"Coordinates must be numeric, got x: {type(self.x).__name__}, y: {type(self.y).__name__}."
            )

    @lru_cache(maxsize=None)
    def distance_to(self, other: "Position") -> float:
        """Calculate the distance to another Position."""
        if not isinstance(other, Position):  #
            raise ValueError("The argument must be a Position instance.")
        return sqrt((self.x - other.x) ** 2 + (self.y - other.y) ** 2)

    def move(self, angle: "Degrees"| int, steps: float) -> "Position":
        """Move the position by a certain number of steps in the specified angle."""
        if not isinstance(angle, Degrees):  #
            raise ValueError("Angle must be a Degrees instance.")

        rad_angle = angle.to_radians()
        return Position(self.x + steps * cos(rad_angle), self.y + steps * sin(rad_angle))

    def is_in(self, top_left: "Position", bottom_right: "Position") -> bool:
        """Check if the position is within a defined rectangular area."""
        if not all(isinstance(p, Position) for p in [top_left, bottom_right]):  #
            raise ValueError("Top left and bottom right must be Position instances.")
        return (
            top_left.x <= self.x <= bottom_right.x
            and bottom_right.y <= self.y <= top_left.y
        )

    def __add__(self, other: "Position") -> "Position":
        """Add two positions."""
        if not isinstance(other, Position):  #
            raise ValueError("The argument must be a Position instance.")
        return Position(self.x + other.x, self.y + other.y)

    def __sub__(self, other: "Position") -> "Position":
        """Subtract two positions."""
        if not isinstance(other, Position):  #
            raise ValueError("The argument must be a Position instance.")
        return Position(self.x - other.x, self.y - other.y)

    def __eq__(self, other: object) -> bool:
        """Check equality of two positions."""
        if not isinstance(other, Position):
            return NotImplemented
        return isclose(self.x, other.x, abs_tol=1e-9) and isclose(
            self.y, other.y, abs_tol=1e-9
        )

    def surrounding(self) -> ["Position"]:
        return [
            (self.x - 1, self.y + 1), (self.x + 1, self.y - 1),  # alt diag
            (self.x, self.y + 1), (self.x, self.y - 1),  # vertical
            (self.x + 1, self.y), (self.x - 1, self.y), # horizontal
            (self.x + 1, self.y + 1), (self.x - 1, self.y - 1),  # main diag
        ]

    def as_tuple(self):
        return self.x, self.y

    def __repr__(self):
        return f"Position(x={self.x}, y={self.y})"

    def __str__(self) -> str:
        return f"({self._format_value(self.x)}, {self._format_value(self.y)})"

    @staticmethod
    def _format_value(value: float) -> str:
        """Format value for string representation."""
        if abs(value - round(value)) < 1e-6:
            return f"{round(value)}"
        else:
            return f"{value:.6f}"
            

In [2]:
# Degrees class

from dataclasses import dataclass, field
from math import radians, isclose, isinf, isnan
from functools import lru_cache

class InvalidAngleError(Exception):
    """Custom exception for invalid angle values."""
    pass

@dataclass(frozen=True)
class Degrees:
    _angle: float = field(repr=False)

    def __post_init__(self):
        # Normalize the angle using the setter


        if not isinstance(self._angle, (int, float)) or isinf(self.angle) or isnan(self.angle):
            raise InvalidAngleError

        object.__setattr__(self, '_angle', self._normalize_angle(self._angle))

    @property
    def angle(self) -> float:
        """Get the normalized angle."""
        return self._angle

    @staticmethod
    @lru_cache(maxsize=None)
    def _normalize_angle(angle: float) -> float:
        """Normalize the angle to be within [0, 360) degrees."""
        return angle % 360

    def turn_left(self, degrees: float = 90) -> 'Degrees':
        """Turn left by the given number of degrees."""
        return Degrees(self._normalize_angle(self.angle - degrees))

    def turn_right(self, degrees: float = 90) -> 'Degrees':
        """Turn right by the given number of degrees."""
        return Degrees(self._normalize_angle(self.angle + degrees))

    def to_radians(self) -> float:
        """Convert degrees to radians."""
        return radians(self.angle)

    def __add__(self, other: 'Degrees') -> 'Degrees':
        """Add two Degrees instances."""
        if not isinstance(other, Degrees): #
            raise ValueError(f"Invalid addition: {self.angle} + {other.angle} results in an invalid angle.")
        return Degrees(self._normalize_angle(self.angle + other.angle))

    def __sub__(self, other: 'Degrees') -> 'Degrees':
        """Subtract two Degrees instances."""
        if not isinstance(other, Degrees): #
            raise ValueError(f"Invalid subtraction: {self.angle} - {other.angle} results in an invalid angle.")

        return Degrees(self._normalize_angle(self.angle - other.angle))

    def __eq__(self, other: object) -> bool:
        """Check equality of two Degrees instances."""
        if not isinstance(other, Degrees):
            return NotImplemented
        return isclose(self.angle, other.angle, abs_tol=1e-9)

    def __hash__(self) -> int:
        """Return a hash of the angle for hashable collections."""
        return hash(round(self.angle, 9))

    def __repr__(self) -> str:
        return f"Degrees(angle={self.angle})"

    def __str__(self) -> str:
        return f"{self.angle}°"

    def to_cardinal(self) -> str:
        """Convert angle to cardinal direction."""
        cardinal_points = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
        index = round(self.angle / 45) % 8
        return cardinal_points[index]
        

In [3]:
# Obstacle class

import random
from dataclasses import dataclass, asdict
from typing import List

#from src.main.world_objects.robot_objects.position import Position


def generate_obstacle_positions(width, height):
    pos_obstacles = []

    n_obstacles = random.randint(1, 10)  # random number of obstacles

    # generate random coordinates within safe zone
    rand = lambda l: random.randint(-l + 5, l + 5)

    x_ranges, y_range = [], []

    rad = 10  # radius of obstacle

    # should draw n obstacles in the safe zone
    while len(pos_obstacles) < n_obstacles:
        x, y = rand(width), rand(height)  # random coordinates in safe zone

        if x not in x_ranges and y not in y_range and (x, y) != (0, 0):
            pos_obstacles.append((x, y))

            x_ranges.extend(list(range(x - rad, x + rad)))
            y_range.extend(list(range(y - rad, y + rad)))

    return pos_obstacles, x_ranges, y_range


def make_obstacle(max_p=10, width=100, height=200, diameter=4) -> list[tuple[range]]:

    # num obstacles:
    n_obstacles = random.randint(0, max_p)

    buffer = 2
    obstacles = []
    while len(obstacles) < n_obstacles:
        x, y = random.randint(
            (-width + diameter + buffer), (width - diameter - buffer)
        ), random.randint((-height + diameter + buffer), (height - diameter - buffer))
        a, b = range(x, x + diameter), range(y, y + diameter)

        is_overlap = any(
            map(lambda _: (overlaps(a, _[0]) and overlaps(b, _[1])), obstacles)
        )
        if not is_overlap:
            # new obstacle doesn't overlaps any others
            obstacles.append((a, b))
    return obstacles


def overlaps(a: range, b: range):
    """Whether a and b overlap."""
    return (a.start <= b.stop) and (b.start <= (a.stop))


def get_obstacles(obstacles):
    """returns the list of obstacles."""
    if obstacles:
        print("There are some obstacles:")
        for a, b in obstacles:
            print(f"- At position {a.start},{b.start} (to {a.stop},{b.stop})")


def is_position_blocked(new_x, new_y, obstacles) -> bool:
    """returns True if position (x,y) falls inside an obstacle."""
    blocked = False
    for _ in obstacles:
        if new_x in _[0] and new_y in _[1]:
            blocked = True
            break
    return blocked


def is_path_blocked(x1, y1, x2, y2, obstacles) -> bool:
    """returns True if there is an obstacle between coordinates (x1, y1) and (x2, y2)."""
    # pick an axis only check that
    a = range(x1, x2) if x2 >= x1 else range(x2, x1)
    b = range(y1, y2) if y2 >= y1 else range(y2, y1)

    blocked = False
    for _ in obstacles:
        if overlaps(a, _[0]) and overlaps(b, _[1]):
            blocked = True
            break
    return blocked


def gen_fence(fence: List[Position], length: int, max_attempts: int = 10) -> List[Position]:
    head = fence[-1]  # Current head position of the fence
    tried_positions = set()  # Set to track tried positions
    attempts = 0  # Track the number of attempts

    while len(fence) < length:
        possible_next = random.choice(head.surrounding())

        # Debug: Print the possible next position
        # print(f"Trying position: {possible_next}")

        # Check if this position is already part of the fence or has been tried
        if possible_next not in [(p.x, p.y) for p in fence] and possible_next not in tried_positions:
            fence.append(Position(possible_next[0], possible_next[1]))  # Add to fence
            # print(f"Added position: {possible_next}")  # Debug: Print added position
            tried_positions.clear()  # Clear tried positions since we found a valid next
            return gen_fence(fence, length, max_attempts)  # Recur to continue building

        # Mark this position as tried
        tried_positions.add(possible_next)
        attempts += 1

        # If maximum attempts reached, break out of the loop
        if attempts >= max_attempts:
            print("No valid next position found, stopping.")
            return fence

    return fence


def visualize_fence(fence: List[Position], grid_size: int = 5):
    grid = [['.' for _ in range(grid_size)] for _ in range(grid_size)]

    for position in fence:
        if 0 <= position.x < grid_size and 0 <= position.y < grid_size:
            grid[position.y][position.x] = '#'

    for row in grid:
        print(" ".join(row))



class Obstacle:

    def __init__(self, position: 'Position'):
        self.position = position
        self.fence = gen_fence([self.position], 5)

    @property
    def position(self) -> Position:
        return self._position
    @position.setter
    def position(self, value):
        self._position = value

    @property
    def fence(self) -> List[Position]:
        return self._fence
    @fence.setter
    def fence(self, value: List[Position]):
        self._fence = value


if __name__ == "__main__":
    # Usage
    starting_position = Position(2, 2)
    fence = [starting_position]
    length = 50  # Length of the fence you want to generate

    fence = gen_fence(fence, length)
    # visualize_fence(fence, 10)


In [4]:
# Robot class

from dataclasses import dataclass, field
from enum import Enum
from typing import Optional

# from src.main.world_objects.robot_objects.fuel_tank import FuelTank
# from src.main.world_objects.robot_objects.position import Position
# from src.main.world_objects.robot_objects.degrees import Degrees
# from src.main.world_objects.robot_objects.shield import Shield
# from src.main.world_objects.robot_objects.weapon import Weapon, WeaponError


class RobotType(Enum):
    SCOUT = "scout"
    SNIPER = "sniper"
    TANK = "tank"
    ASSAULT = "assault"
    SUPPORT = "support"


@dataclass
class Robot:
    name: str = field(default="bot")
    position: Position = field(default_factory=lambda: Position(0, 0))
    direction: Degrees = field(default_factory=lambda: Degrees(0))
    shield: Shield = field(default_factory=lambda: Shield(shield_max=5))
    weapon: Weapon = field(default_factory=lambda: Weapon(_ammo=5))
    tank: FuelTank = field(default_factory=lambda: FuelTank(volume=50))
    robot_type: RobotType = field(default=RobotType.SUPPORT)

    def __post_init__(self):
        self._set_attributes_based_on_type()

    def move_forward(self, nr_steps: int) -> bool:
        return self._update_position(nr_steps, forward=True)

    def move_backward(self, nr_steps: int) -> bool:
        return self._update_position(nr_steps, forward=False)

    def turn_left(self, degrees: float = 90) -> None:
        return Robot(name=self.name, position=self.position, direction=self.direction.turn_left(degrees), robot_type=self.robot_type, tank=self.tank)
        
    def turn_right(self, degrees: float = 90) -> None:
        return Robot(name=self.name, position=self.position, direction=self.direction.turn_right(degrees), robot_type=self.robot_type, tank=self.tank)

    def damage_shield(self, damage: float) -> None:
        return Robot(name=self.name, position=self.position, direction=self.direction, robot_type=self.robot_type, tank=self.tank, shield=self.shield.damage_shield(damage))

    def repair_shield(self) -> None:
        return Robot(name=self.name, position=self.position, direction=self.direction, robot_type=self.robot_type, tank=self.tank, shield=self.shield.repair_shield())
        
    def shoot(self) -> None:
        try:
            return Robot(name=self.name, position=self.position, direction=self.direction, robot_type=self.robot_type, tank=self.tank, weapon=self.weapon.shot())
        except WeaponError as e:
            print(f"{e}")

    def reload(self) -> None:
        try:
            return Robot(name=self.name, position=self.position, direction=self.direction, robot_type=self.robot_type, tank=self.tank, weapon=self.weapon.reload())
        except WeaponError as e:
            print(e)

    def refuel(self) -> None:
        try:
            return Robot(name=self.name, position=self.position, direction=self.direction, robot_type=self.robot_type, tank=self.tank.refuel(), weapon=self.weapon)
        except ValueError as e:
            print(e)

    def shield_level(self) -> Optional[int]:
        return self.shield.level

    def tank_level(self) -> float:
        return self.tank.level

    def _update_position(self, nr_steps: int, forward: bool) -> bool:
        steps = nr_steps if forward else -nr_steps
        
        if self._drop_fuel(nr_steps): 
            return Robot(name=self.name, position=self.position.move(self.direction, steps), direction=self.direction,robot_type=self.robot_type, tank=self.tank)
        return self # return the new robot with the updated position since all objects have to be hashable (frozen dataclasses)
        

    def _drop_fuel(self, steps: int):
        try:
            return Robot(name=self.name, position=self.position, direction=self.direction, robot_type=self.robot_type, tank=self.tank.drop_fuel(distance=steps))
        except ValueError:
            print("not enough fuel")
            return self

    
    def _set_attributes_based_on_type(self):
        # Define attributes based on the RobotType Enum
        type_attributes = {
            RobotType.SCOUT: (1, 1, 1, 1, 1),
            RobotType.SNIPER: (2, 2, 2, 2, 2),
            RobotType.TANK: (3, 3, 3, 3, 3),
            RobotType.ASSAULT: (4, 4, 4, 4, 4),
            RobotType.SUPPORT: (5, 5, 5, 5, 5),
        }

        # Normalize robot_type first
        if isinstance(self.robot_type, str):
            try:
                self.robot_type = RobotType(self.robot_type.lower())
            except ValueError:
                allowed = [t.value for t in RobotType]
                print(f"Invalid robot type: '{self.robot_type}'. Must be one of: {allowed}")
                self.robot_type = RobotType.SUPPORT

        # Look up attributes based on the robot's type
        attributes = type_attributes.get(self.robot_type)
        if attributes:
            shot_damage, ammo_max, shield_max, repair_delay, reload_delay = attributes
            self.weapon = Weapon(
                _ammo=ammo_max, _load_delay=reload_delay, _damage=shot_damage, _ammo_max=ammo_max)
            self.shield = Shield(shield_max=shield_max,
                                 repair_delay=repair_delay)

    def __str__(self):
        return (
            f"Name: {self.name}, Position: {self.position}, Direction: {self.direction.angle}, "
            f"Shield Level: {self.shield_level()}, Ammo: {self.weapon.ammo}, Fuel Level: {self.tank_level()}, Type: {self.robot_type.value}"
        )
